# Tutorial 2: Simulated Data Workflow

This tutorial demonstrates **parameter recovery** using Monte Carlo simulation.
We'll set known "true" parameters, generate synthetic data, estimate the parameters,
and verify that we can recover the truth.

## Why Simulate?

Simulation studies are essential for:

1. **Validation**: Verify the estimator works correctly
2. **Understanding identification**: See what sample sizes are needed
3. **Debugging**: When real data estimates seem wrong, simulation helps diagnose
4. **Power analysis**: Determine if your data can detect economically meaningful effects

## The Experiment

We'll:
1. Set true parameters $(\theta_c^*, RC^*)$
2. Generate data from the model
3. Estimate parameters (pretending we don't know the truth)
4. Compare estimates to true values
5. Visualize parameter recovery

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

from econirl import RustBusEnvironment, LinearUtility, NFXPEstimator
from econirl.simulation import simulate_panel

print("econirl loaded successfully!")

## Step 1: Set True Parameters

We choose parameter values that are economically meaningful:
- Operating cost: Higher mileage = higher maintenance costs
- Replacement cost: Substantial fixed cost for new engine

In [ ]:
# True parameters (what we'll try to recover)
TRUE_OPERATING_COST = 0.001
TRUE_REPLACEMENT_COST = 3.0

print("True Parameters")
print("=" * 40)
print(f"Operating cost (theta_c): {TRUE_OPERATING_COST}")
print(f"Replacement cost (RC):    {TRUE_REPLACEMENT_COST}")

In [ ]:
# Create environment with true parameters
env = RustBusEnvironment(
    operating_cost=TRUE_OPERATING_COST,
    replacement_cost=TRUE_REPLACEMENT_COST,
    num_mileage_bins=90,
    discount_factor=0.9999,
    scale_parameter=1.0,
    seed=42,
)

print(env.describe())

## Step 2: Generate Synthetic Data

We simulate a panel of buses making replacement decisions according to the
optimal policy implied by the true parameters.

In [ ]:
# Simulation settings
N_INDIVIDUALS = 500
N_PERIODS = 100
SEED = 12345

# Generate data
panel = simulate_panel(
    env,
    n_individuals=N_INDIVIDUALS,
    n_periods=N_PERIODS,
    seed=SEED,
)

print(f"Generated Panel:")
print(f"  Individuals: {panel.num_individuals}")
print(f"  Periods: {N_PERIODS}")
print(f"  Total observations: {panel.num_observations:,}")

In [ ]:
# Examine the simulated data
all_states = panel.get_all_states().numpy()
all_actions = panel.get_all_actions().numpy()

print(f"\nData Summary:")
print(f"  Replacement rate: {all_actions.mean():.2%}")
print(f"  Mean mileage bin: {all_states.mean():.1f}")
print(f"  Max mileage bin: {all_states.max()}")

In [ ]:
# Visualize simulated data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State distribution
axes[0].hist(all_states, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Mileage Bin')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Simulated States')

# Empirical replacement rates
states_unique = np.arange(env.num_states)
empirical_rates = np.zeros(env.num_states)
counts = np.zeros(env.num_states)

for s in states_unique:
    mask = all_states == s
    counts[s] = mask.sum()
    if counts[s] > 0:
        empirical_rates[s] = all_actions[mask].mean()

valid = counts >= 10
axes[1].scatter(states_unique[valid], empirical_rates[valid],
                s=counts[valid]/10, alpha=0.6, color='coral')
axes[1].set_xlabel('Mileage Bin')
axes[1].set_ylabel('Replacement Rate')
axes[1].set_title('Empirical Replacement Rate\n(from simulated data)')

plt.tight_layout()
plt.show()

## Step 3: Estimate Parameters

Now we pretend we don't know the true parameters and estimate them from data.

In [ ]:
# Set up estimation
utility = LinearUtility.from_environment(env)
estimator = NFXPEstimator(se_method="asymptotic", verbose=True)

# Run estimation
result = estimator.estimate(
    panel=panel,
    utility=utility,
    problem=env.problem_spec,
    transitions=env.transition_matrices,
)

print("\nEstimation complete!")

## Step 4: Compare Estimates to Truth

In [ ]:
# Full summary
print(result.summary())

In [ ]:
# Detailed comparison
true_params = env.get_true_parameter_vector()

print("Parameter Recovery Analysis")
print("=" * 60)

recovery_results = []
for i, name in enumerate(result.parameter_names):
    true_val = true_params[i].item()
    est_val = result.parameters[i].item()
    se = result.standard_errors[i].item()
    
    bias = est_val - true_val
    pct_bias = (bias / true_val) * 100 if true_val != 0 else float('inf')
    t_stat = bias / se if se > 0 else float('inf')
    
    # Is true value in 95% CI?
    ci_low, ci_high = result.confidence_interval(alpha=0.05)
    in_ci = ci_low[i].item() <= true_val <= ci_high[i].item()
    
    recovery_results.append({
        'name': name,
        'true': true_val,
        'estimated': est_val,
        'se': se,
        'bias': bias,
        'pct_bias': pct_bias,
        'in_ci': in_ci,
    })
    
    print(f"\n{name}:")
    print(f"  True value:      {true_val:.6f}")
    print(f"  Estimated:       {est_val:.6f}")
    print(f"  Std Error:       {se:.6f}")
    print(f"  Bias:            {bias:+.6f} ({pct_bias:+.1f}%)")
    print(f"  True in 95% CI:  {'Yes' if in_ci else 'No'}")

In [ ]:
# Visualize parameter recovery
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, res in enumerate(recovery_results):
    ax = axes[idx]
    
    ci_low, ci_high = result.confidence_interval(alpha=0.05)
    
    # Plot estimate with CI
    ax.errorbar(
        0, res['estimated'],
        yerr=[[res['estimated'] - ci_low[idx].item()], 
              [ci_high[idx].item() - res['estimated']]],
        fmt='o', markersize=12, capsize=8, capthick=2,
        color='steelblue', ecolor='steelblue',
        label='Estimated (95% CI)'
    )
    
    # Plot true value
    ax.axhline(res['true'], color='coral', linestyle='--', 
               linewidth=2, label='True value')
    
    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
    ax.set_ylabel('Parameter Value')
    ax.set_title(f"{res['name']}\nTrue: {res['true']:.4f}, Est: {res['estimated']:.4f}")
    ax.legend(loc='best')

plt.suptitle('Parameter Recovery with 95% Confidence Intervals', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 5: Sample Size Effects

Let's see how estimation precision improves with sample size.

In [ ]:
# Run estimation at different sample sizes
sample_sizes = [100, 250, 500, 1000]
results_by_size = []

for n in sample_sizes:
    # Generate data
    panel_n = simulate_panel(env, n_individuals=n, n_periods=100, seed=SEED)
    
    # Estimate
    result_n = NFXPEstimator(se_method="asymptotic", verbose=False).estimate(
        panel=panel_n,
        utility=utility,
        problem=env.problem_spec,
        transitions=env.transition_matrices,
    )
    
    results_by_size.append({
        'n': n,
        'obs': panel_n.num_observations,
        'params': result_n.parameters.numpy(),
        'se': result_n.standard_errors.numpy(),
    })
    
    print(f"n={n:4d}: theta_c={result_n.parameters[0].item():.6f} "
          f"(SE={result_n.standard_errors[0].item():.6f}), "
          f"RC={result_n.parameters[1].item():.4f} "
          f"(SE={result_n.standard_errors[1].item():.4f})")

In [ ]:
# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_vals = [r['n'] for r in results_by_size]
true_vals = [TRUE_OPERATING_COST, TRUE_REPLACEMENT_COST]
param_names = ['Operating Cost', 'Replacement Cost']

for idx, (ax, name, true_val) in enumerate(zip(axes, param_names, true_vals)):
    estimates = [r['params'][idx] for r in results_by_size]
    ses = [r['se'][idx] for r in results_by_size]
    
    ax.errorbar(n_vals, estimates, yerr=[1.96*s for s in ses],
                fmt='o-', markersize=8, capsize=5, capthick=2,
                color='steelblue', label='Estimate +/- 1.96 SE')
    ax.axhline(true_val, color='coral', linestyle='--', 
               linewidth=2, label='True value')
    
    ax.set_xlabel('Number of Individuals')
    ax.set_ylabel('Parameter Estimate')
    ax.set_title(f'{name}')
    ax.legend()

plt.suptitle('Convergence with Sample Size', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

This tutorial demonstrated:

1. **Setting up a simulation study** with known true parameters
2. **Generating synthetic data** using `simulate_panel()`
3. **Estimating parameters** and comparing to ground truth
4. **Verifying coverage** - the true value should be in the 95% CI about 95% of the time
5. **Understanding sample size effects** - larger samples yield more precise estimates

### Key Takeaways

- **Parameter recovery works**: The estimator correctly recovers true parameters
- **Standard errors are valid**: Confidence intervals have correct coverage
- **More data helps**: Precision improves with $\sqrt{n}$ as theory predicts

### Next Steps

- Try different true parameter values
- Run multiple replications to verify coverage rates
- See [Tutorial 3](real_data_example.ipynb) for real data applications